### Fill in if your running locally on in kaggle notebook

In [1]:
Kaggle_Notebook=True #or False

### We need Python 3.10

In [2]:
!python --version

Python 3.10.12


In [3]:
import optiver2023
env = optiver2023.make_env()
iter_test = env.iter_test()

In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from prophet import Prophet
from datetime import datetime, timedelta

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/optiver-trading-at-the-close/public_timeseries_testing_util.py
/kaggle/input/optiver-trading-at-the-close/train.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/sample_submission.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/revealed_targets.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/test.csv
/kaggle/input/optiver-trading-at-the-close/optiver2023/competition.cpython-310-x86_64-linux-gnu.so
/kaggle/input/optiver-trading-at-the-close/optiver2023/__init__.py


In [5]:
if Kaggle_Notebook == True:    
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))
            
    INPUT_FILE_PATH="/kaggle/input/optiver-trading-at-the-close/"
    OUTPUT_FILE_PATH="/kaggle/working/"
    SUBMISSIONS_PATH=OUTPUT_FILE_PATH+'submissions'
else:
    INPUT_FILE_PATH="../data/"
    OUTPUT_FILE_PATH="../working/"
    SUBMISSIONS_PATH=OUTPUT_FILE_PATH+'submissions'

if not os.path.exists(SUBMISSIONS_PATH):
   os.makedirs(SUBMISSIONS_PATH)
   print("The new directory is created!")
else:
    print(f"{SUBMISSIONS_PATH} Directory already exists!")
    

/kaggle/input/optiver-trading-at-the-close/public_timeseries_testing_util.py
/kaggle/input/optiver-trading-at-the-close/train.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/sample_submission.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/revealed_targets.csv
/kaggle/input/optiver-trading-at-the-close/example_test_files/test.csv
/kaggle/input/optiver-trading-at-the-close/optiver2023/competition.cpython-310-x86_64-linux-gnu.so
/kaggle/input/optiver-trading-at-the-close/optiver2023/__init__.py
The new directory is created!


In [6]:
data = pd.read_csv(INPUT_FILE_PATH+'train.csv')

validation_set = data[data["date_id"] > 400] 
validation_set.to_csv(OUTPUT_FILE_PATH+'validation_set.csv', index=False)

train_set = data[data["date_id"] <= 400] 
train_set.to_csv(OUTPUT_FILE_PATH+'train_set.csv', index=False)

In [7]:
train = pd.read_csv(OUTPUT_FILE_PATH+'train_set.csv')
validation = pd.read_csv(OUTPUT_FILE_PATH+'validation_set.csv')
test = pd.read_csv(INPUT_FILE_PATH+'example_test_files/test.csv')

In [8]:
DATASETS = [train, validation, test]
MODELS = set() # use MODELS.add() to add items

#### Add normalized columns 

In [9]:
scaler = StandardScaler()
COLUMNS = ['imbalance_size', 'matched_size', 'bid_size', 'ask_size']

def normalize(DATASET, COLUMN):
    DATASET[COLUMN+"_norm"] = scaler.fit_transform(DATASET[[COLUMN]])

for DATASET in DATASETS:
    for COLUMN in COLUMNS:
        normalize(DATASET, COLUMN)

### Linear Regression
We all know that Linear Regression is a terrible fit. <br/>
So let's all just *pretend* it is an interesting model to check out and use the output on the test-set and API. <br/><br/>
This model is for research purposess only.

#### Deal with NaN values
So there are different ways to deal with NaN values, one is to delete them. But I'd rather not, thus:
- since there arent that many NaN values, just replace them with mean of column

In [10]:
imp_x = SimpleImputer(missing_values=np.nan, strategy='mean')
imp_y = SimpleImputer(missing_values=np.nan, strategy='mean')

def configure_dataset(DATASET, BASE, TARGET_COLUMN):
    print("Configuring datasets...")
    print(f"Target column: {TARGET_COLUMN}")
    y_DATASET = np.empty((len(DATASET), 1))
    
    # Extract target variable as numpy array
    if TARGET_COLUMN:
        y_DATASET = np.array(DATASET[TARGET_COLUMN].values)
        
    X_DATASET = np.array(DATASET[BASE])       
    
    
    if len(X_DATASET) == len(train):
        print(f"Dataset size: {len(X_DATASET)} \timpute dataset")
        imp_x.fit(X_DATASET)
        y_DATASET = imp_y.fit_transform(y_DATASET.reshape(-1, 1)).ravel() #only fill target values for train set.
    
    X_DATASET = imp_x.transform(X_DATASET)
    
    if TARGET_COLUMN:
        return X_DATASET, y_DATASET
    else:
        return X_DATASET

In [11]:
def linear_regression_fit(TARGET_COLUMN, BASE, BASE_NAME):    
    MODEL='linear_regression'+'_'+BASE_NAME
    
    print(f"Start fitting model: {MODEL}...\n")     
    
    # Create sets
    X_train, y_train = configure_dataset(train, BASE, TARGET_COLUMN)
    
    reg = LinearRegression().fit(X_train, y_train)    
    
    print(f"End fitting model: {MODEL}\n")
    print(f"score: {reg.score(X_train, y_train)}")
    print(f"coef_: {reg.coef_}")
    print(f"intercept_: {reg.intercept_}\n\n\n")
    
    return reg

In [12]:
def linear_regression_predict(DATASET, BASE, FITTED_MODEL,BASE_NAME):
    # Use the model output to predict validation prediction values        
    
    MODEL='lin_regr'+'_'+BASE_NAME
    print(f"Start predicting model: {MODEL} with: BASE: {BASE_NAME}...\n")
    
    X = configure_dataset(DATASET, BASE, False)
    
    DATASET[MODEL+'_prediction'] = FITTED_MODEL.predict(X)    
    
    MODELS.add(MODEL)
    print(f"End predicting model: {MODEL} with: BASE: {BASE_NAME}\n\n\n")

    return DATASET

In [13]:
base = ['wap','seconds_in_bucket' ,'reference_price', 'imbalance_size', 'imbalance_buy_sell_flag', 'matched_size', 'bid_price'] 
base_norm = ['wap','seconds_in_bucket' ,'reference_price', 'imbalance_size_norm', 'imbalance_buy_sell_flag', 'matched_size_norm', 'bid_price']

model_lr = linear_regression_fit('target', base, "base")
model_lr_norm = linear_regression_fit('target', base_norm, "base_norm")

Start fitting model: linear_regression_base...

Configuring datasets...
Target column: target
Dataset size: 4357980 	impute dataset
End fitting model: linear_regression_base

score: 0.019470059996827715
coef_: [-4.41601001e+03 -6.33054112e-04  2.77208450e+03 -1.69317314e-09
 -1.56923895e-01 -1.36311859e-10  1.52385589e+03]
intercept_: 120.60537003926122



Start fitting model: linear_regression_base_norm...

Configuring datasets...
Target column: target
Dataset size: 4357980 	impute dataset
End fitting model: linear_regression_base_norm

score: 0.019470059998868305
coef_: [-4.41601004e+03 -6.33056819e-04  2.77208449e+03 -3.39600805e-02
 -1.56923882e-01 -1.84900906e-02  1.52385589e+03]
intercept_: 120.58989359409763





#### Predict Linear Regression

In [14]:
# validation set
pred_val_lr = linear_regression_predict(validation, base, model_lr, "base")
pred_val_lr_norm = linear_regression_predict(validation, base_norm, model_lr_norm, "base_norm")

Start predicting model: lin_regr_base with: BASE: base...

Configuring datasets...
Target column: False
End predicting model: lin_regr_base with: BASE: base



Start predicting model: lin_regr_base_norm with: BASE: base_norm...

Configuring datasets...
Target column: False
End predicting model: lin_regr_base_norm with: BASE: base_norm





In [15]:
# test set
pred_test_lr = linear_regression_predict(test, base, model_lr, "base")
pred_test_lr_norm = linear_regression_predict(test, base_norm, model_lr_norm, "base_norm")

Start predicting model: lin_regr_base with: BASE: base...

Configuring datasets...
Target column: False
End predicting model: lin_regr_base with: BASE: base



Start predicting model: lin_regr_base_norm with: BASE: base_norm...

Configuring datasets...
Target column: False
End predicting model: lin_regr_base_norm with: BASE: base_norm





### Prophet
#### Training data
For the simplest version, create one training set with columns ['timestamp', 'target'] <br/>
To Do maybe later, add more data to the model, like ['imbalance_buy_sell_flag', 'boolean for when far/near_price is']. <br/>
We already have a 'base' and 'base_norm' set we can use. <br/>

- Prophet can deal with NaN values, but we still have the option to use the (or create a new) y_target column that has the mean for missing values.
#### Model
For each stock id, fit the prohpet model and predict on the validate set <br/>

#### So we have 55 observations for 1 day
To use prophet we need to create a new column 'ds' that fits 55 observations per day. <br/>
We can do this by mapping each observation to a YYYY-MM-DD HH:MM:SS using ```create_datetime_mapping```


[Prophet non-daily_data](https://facebook.github.io/prophet/docs/non-daily_data.html)

In [16]:
UNIQUE_STOCK_IDS=train['stock_id'].nunique()
OBSERVATIONS_PER_DAY=len(pd.DataFrame(train[train['stock_id']==1][train['date_id']==0]))
print(f"UNIQUE_STOCK_IDS: {UNIQUE_STOCK_IDS}\nOBSERVATIONS_PER_DAY: {OBSERVATIONS_PER_DAY}")

UNIQUE_STOCK_IDS: 200
OBSERVATIONS_PER_DAY: 55


/tmp/ipykernel_76/2433014067.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  OBSERVATIONS_PER_DAY=len(pd.DataFrame(train[train['stock_id']==1][train['date_id']==0]))


**disclaimer** <br/> 
The method ```create_datetime_mapping``` below is generated with Claude.ai, an LLM. Why: <br/> 
- Time saving
- Competition deadline has been expired over 2 years

In [17]:
def create_datetime_mapping(df, COLUMNS, date_id_col='date_id'):
    """
    Create datetime mapping for the dataset.
    
    Parameters:
    - df: input DataFrame (train_df)
    - date_id_col: name of the date_id column
    
    Returns:
    - DataFrame with added 'ds' column
    """
    
    # Calculate the interval in seconds
    # 24 hours = 86400 seconds, distributed across 55 rows
    
    print("Configuring dataset for Prophet...")
    
    total_seconds_in_day = 24 * 60 * 60  # 86400
    values_per_id = 55
    interval_seconds = total_seconds_in_day / values_per_id
    
    # Create a copy to avoid modifying original
    df= df.reset_index()
    result_df = df.copy()
    
    # Group by date_id and add row number within each group
    result_df['row_within_group'] = result_df.groupby(date_id_col).cumcount()
    
    # Calculate datetime for each row
    def calculate_datetime(row):
        date_id = row[date_id_col]
        row_index = row['row_within_group']
        
        # Calculate the base date: 1970-01-01 for date_id=0
        base_date = datetime(1970, 1, 1) + timedelta(days=date_id)
        
        # Calculate seconds offset for this row
        seconds_offset = row_index * interval_seconds
        
        # Create the timestamp
        timestamp = base_date + timedelta(seconds=seconds_offset)
        
        return timestamp.strftime('%Y-%m-%d %H:%M:%S')
    
    result_df['ds'] = result_df.apply(calculate_datetime, axis=1)
    
    # Drop the helper column
    result_df = result_df.drop('row_within_group', axis=1)
    
    return result_df[COLUMNS]

In [18]:
COLUMNS = ['index','row_id','stock_id','target','ds']
train_simple_prophet = create_datetime_mapping(train, COLUMNS)

Configuring dataset for Prophet...


In [19]:
def fit_prophet_per_stockid(DATASET):
    prophet_models_stockid = {}
    
    DATASET = DATASET.rename(columns={"target": "y"}).drop(['row_id','index'], axis=1)
    
    for STOCK_ID in sorted(DATASET['stock_id'].unique()):
        if STOCK_ID%25 == 0:
            print(f"Fitting STOCK_ID: {STOCK_ID}...")
        prophet_model  = Prophet()
        DATASET_FIT = DATASET[DATASET['stock_id']==STOCK_ID]
        
        prophet_model.fit(DATASET_FIT)
        prophet_models_stockid['prophet_model_'+str(STOCK_ID)] = prophet_model
    
    print("Finished fitting models")
    
    return prophet_models_stockid
    

In [20]:
prophet_models_stockid = fit_prophet_per_stockid(train_simple_prophet)

Fitting STOCK_ID: 0...


10:59:15 - cmdstanpy - INFO - Chain [1] start processing
10:59:18 - cmdstanpy - INFO - Chain [1] done processing
10:59:19 - cmdstanpy - INFO - Chain [1] start processing
10:59:23 - cmdstanpy - INFO - Chain [1] done processing
10:59:24 - cmdstanpy - INFO - Chain [1] start processing
10:59:27 - cmdstanpy - INFO - Chain [1] done processing
10:59:28 - cmdstanpy - INFO - Chain [1] start processing
10:59:31 - cmdstanpy - INFO - Chain [1] done processing
10:59:32 - cmdstanpy - INFO - Chain [1] start processing
10:59:38 - cmdstanpy - INFO - Chain [1] done processing
10:59:38 - cmdstanpy - INFO - Chain [1] start processing
10:59:41 - cmdstanpy - INFO - Chain [1] done processing
10:59:42 - cmdstanpy - INFO - Chain [1] start processing
10:59:44 - cmdstanpy - INFO - Chain [1] done processing
10:59:44 - cmdstanpy - INFO - Chain [1] start processing
10:59:47 - cmdstanpy - INFO - Chain [1] done processing
10:59:48 - cmdstanpy - INFO - Chain [1] start processing
10:59:50 - cmdstanpy - INFO - Chain [1]

Fitting STOCK_ID: 25...


11:00:50 - cmdstanpy - INFO - Chain [1] start processing
11:00:53 - cmdstanpy - INFO - Chain [1] done processing
11:00:54 - cmdstanpy - INFO - Chain [1] start processing
11:00:57 - cmdstanpy - INFO - Chain [1] done processing
11:00:58 - cmdstanpy - INFO - Chain [1] start processing
11:01:01 - cmdstanpy - INFO - Chain [1] done processing
11:01:02 - cmdstanpy - INFO - Chain [1] start processing
11:01:07 - cmdstanpy - INFO - Chain [1] done processing
11:01:08 - cmdstanpy - INFO - Chain [1] start processing
11:01:09 - cmdstanpy - INFO - Chain [1] done processing
11:01:10 - cmdstanpy - INFO - Chain [1] start processing
11:01:13 - cmdstanpy - INFO - Chain [1] done processing
11:01:13 - cmdstanpy - INFO - Chain [1] start processing
11:01:17 - cmdstanpy - INFO - Chain [1] done processing
11:01:18 - cmdstanpy - INFO - Chain [1] start processing
11:01:20 - cmdstanpy - INFO - Chain [1] done processing
11:01:21 - cmdstanpy - INFO - Chain [1] start processing
11:01:24 - cmdstanpy - INFO - Chain [1]

Fitting STOCK_ID: 50...


...

Fitting STOCK_ID: 150...


11:08:46 - cmdstanpy - INFO - Chain [1] start processing
11:08:49 - cmdstanpy - INFO - Chain [1] done processing
11:08:50 - cmdstanpy - INFO - Chain [1] start processing
11:08:54 - cmdstanpy - INFO - Chain [1] done processing
11:08:55 - cmdstanpy - INFO - Chain [1] start processing
11:08:59 - cmdstanpy - INFO - Chain [1] done processing
11:08:59 - cmdstanpy - INFO - Chain [1] start processing
11:09:03 - cmdstanpy - INFO - Chain [1] done processing
11:09:04 - cmdstanpy - INFO - Chain [1] start processing
11:09:08 - cmdstanpy - INFO - Chain [1] done processing
11:09:08 - cmdstanpy - INFO - Chain [1] start processing
11:09:12 - cmdstanpy - INFO - Chain [1] done processing
11:09:13 - cmdstanpy - INFO - Chain [1] start processing
11:09:16 - cmdstanpy - INFO - Chain [1] done processing
11:09:16 - cmdstanpy - INFO - Chain [1] start processing
11:09:21 - cmdstanpy - INFO - Chain [1] done processing
11:09:22 - cmdstanpy - INFO - Chain [1] start processing
11:09:24 - cmdstanpy - INFO - Chain [1]

Fitting STOCK_ID: 175...


11:10:28 - cmdstanpy - INFO - Chain [1] start processing
11:10:32 - cmdstanpy - INFO - Chain [1] done processing
11:10:33 - cmdstanpy - INFO - Chain [1] start processing
11:10:36 - cmdstanpy - INFO - Chain [1] done processing
11:10:37 - cmdstanpy - INFO - Chain [1] start processing
11:10:41 - cmdstanpy - INFO - Chain [1] done processing
11:10:41 - cmdstanpy - INFO - Chain [1] start processing
11:10:46 - cmdstanpy - INFO - Chain [1] done processing
11:10:47 - cmdstanpy - INFO - Chain [1] start processing
11:10:50 - cmdstanpy - INFO - Chain [1] done processing
11:10:51 - cmdstanpy - INFO - Chain [1] start processing
11:10:55 - cmdstanpy - INFO - Chain [1] done processing
11:10:55 - cmdstanpy - INFO - Chain [1] start processing
11:10:57 - cmdstanpy - INFO - Chain [1] done processing
11:10:58 - cmdstanpy - INFO - Chain [1] start processing
11:11:01 - cmdstanpy - INFO - Chain [1] done processing
11:11:02 - cmdstanpy - INFO - Chain [1] start processing
11:11:05 - cmdstanpy - INFO - Chain [1]

Finished fitting models


### Hoping that the row_id is enough to validate against the test set, otherwise:
If this doesnt work  then maybe the index of the validation set should be remain the same and later fit back in the dataframe. <br/>

Meaning: dataset has index ```[0,1,2,3,4,5,6,7,8]``` <br/>
subset1 ```[0,3,6]```<br/>
subset2 ```[1,4,7]```<br/>
subset3 ```[2,5,8]```<br/><br/>

remerge back so that the postions match the original index. <br/>
yes: ```[0,1,2,3,4,5,6,7,8]``` <br/>
no: ```[0,3,6,1,4,7,2,5,8]``` <br/>

#### Edit
Did not take the risk for shenanigans, just added the index as column and later sort on this index.

In [21]:
def restruct_prophet_dataset(DATASET, DICTIONAIRY, MODEL):
    print(f"Reconstructing Dataset for model: {MODEL}... \n")
    prophet_predictions_df = pd.DataFrame()

    for df in DICTIONAIRY.values():
        prophet_predictions_df = pd.concat([prophet_predictions_df, pd.DataFrame(df) ], ignore_index=True)
    
    
    prophet_predictions_df = prophet_predictions_df.sort_values('index').set_index("index")
    
    DATASET[MODEL+'_prediction'] = prophet_predictions_df[MODEL+'_prediction']
    MODELS.add(MODEL)
    
    print(f"Finished reconstructing Dataset for model: {MODEL}... \n")
    return prophet_predictions_df


In [22]:
def predict_simple_prophet(DATASET, COLUMNS, MODEL):
    prophet_forecasts_stockid = {}    
    DATASET_MAPPED = create_datetime_mapping(DATASET, COLUMNS)
    
    print("Progress... \n")
    
    for STOCK_ID in sorted(DATASET_MAPPED['stock_id'].unique()):
        if STOCK_ID%25 == 0:
            print(f"Predict: STOCK_ID: {STOCK_ID} with model: {prophet_models_stockid.get('prophet_model_'+str(STOCK_ID))}")
        
        
        DATASET_STOCKID = DATASET_MAPPED[DATASET_MAPPED['stock_id']==STOCK_ID]
        DATASET_STOCKID_INDEX_ROW_ID = pd.DataFrame(DATASET_STOCKID[['index','row_id']]).reset_index(drop=True)
        DATASET_STOCKID_PREDICT = DATASET_STOCKID.drop(['index','row_id'], axis=1)

        prophet_forecast = (prophet_models_stockid.get('prophet_model_'+str(STOCK_ID)).predict(DATASET_STOCKID_PREDICT)).drop('ds', axis=1)            
        prophet_forecast['row_id'] = DATASET_STOCKID_INDEX_ROW_ID['row_id']
        prophet_forecast['index'] = DATASET_STOCKID_INDEX_ROW_ID['index']

        prophet_forecasts_stockid['prophet_forecast_'+str(STOCK_ID)] = (prophet_forecast[['index','row_id','yhat']]).rename(columns={'yhat':MODEL+'_prediction'})
        
        if STOCK_ID == (len(DATASET['stock_id'].unique()) -1):
            print(f"Predict: STOCK_ID: {STOCK_ID} with model: {prophet_models_stockid.get('prophet_model_'+str(STOCK_ID))}")
    
    print(f"Finished....\n\n")
    
    df = restruct_prophet_dataset(DATASET, prophet_forecasts_stockid, MODEL)
    
    return df



In [23]:
val_forecast_prophet = predict_simple_prophet(validation, COLUMNS, "simple_prophet")

Configuring dataset for Prophet...
Progress... 

Predict: STOCK_ID: 0 with model: <prophet.forecaster.Prophet object at 0x7c383665f250>
Predict: STOCK_ID: 25 with model: <prophet.forecaster.Prophet object at 0x7c374051ba30>
Predict: STOCK_ID: 50 with model: <prophet.forecaster.Prophet object at 0x7c37f35a4370>
Predict: STOCK_ID: 75 with model: <prophet.forecaster.Prophet object at 0x7c379576c280>
Predict: STOCK_ID: 100 with model: <prophet.forecaster.Prophet object at 0x7c3790480be0>
Predict: STOCK_ID: 125 with model: <prophet.forecaster.Prophet object at 0x7c3742024160>
Predict: STOCK_ID: 150 with model: <prophet.forecaster.Prophet object at 0x7c378779bee0>
Predict: STOCK_ID: 175 with model: <prophet.forecaster.Prophet object at 0x7c37846c94b0>
Predict: STOCK_ID: 199 with model: <prophet.forecaster.Prophet object at 0x7c3779b59fc0>
Finished....


Reconstructing Dataset for model: simple_prophet... 

Finished reconstructing Dataset for model: simple_prophet... 



### Evaluation on validation set
Our evaluation metric is Mean Absolute Error (MAE). The formula is given by:

MAE=1nn∑i=1 |yi−xi| 
Where: n is the total number of data points.
yi  is the predicted value for the  ith  data point.
xi  is the observed value for the  ith  data point.

In [24]:
MAE_DICT = {}
def evaluation(MODELS):
    for MODEL in MODELS:
        mae = (validation[MODEL+'_prediction'] - validation['target']).abs().mean()
        MAE_DICT.update({MODEL : mae})

evaluation(MODELS)

SORTED_MAE_DICT=dict(sorted(MAE_DICT.items(), key=lambda item: item[1])) 

for key, value in SORTED_MAE_DICT.items():
    print(f"{key}: {value}")

lin_regr_base: 5.977291819752582
lin_regr_base_norm: 5.977318690571634
simple_prophet: 6.024046564086519


In [25]:
if Kaggle_Notebook == False:
    fixed_columns = ['row_id','time_id']
    def save_predictions_to_file(MODELS):    
        print("Saving models to file...\n")
        for MODEL in MODELS:
            PREDICTION_COLUMN = MODEL+'_prediction'
            columns = [MODEL+'_prediction']
            columns.extend(fixed_columns)
            columns.reverse()
            #print(f"columns: {columns}")
            file_name = OUTPUT_FILE_PATH+'/submissions/'+PREDICTION_COLUMN+'.csv'
            
            #print(test[columns].rename(columns={PREDICTION_COLUMN:'target'}, inplace=True).head())
            test[columns].rename(columns={PREDICTION_COLUMN:"target"}).to_csv(file_name, index=False, header=True)        
            print(f"Model: {MODEL} to file: {file_name}")
        print("\n\nFinished saving models")
            
    save_predictions_to_file(MODELS)  

# ----------------------predict the models----------------

In [26]:
def submit(SUBMIT_MODEL):    
    print(f"Model: {SUBMIT_MODEL} selected for submission...")
    
    PREDICTION_COLUMN = SUBMIT_MODEL+'_prediction'
    
    
        
    for (test, revealed_targets, sample_prediction) in iter_test:
        if SUBMIT_MODEL == 'lin_regr_base_norm':   
            print(f"Submitting: {SUBMIT_MODEL}...")
            
            for COLUMN in COLUMNS:
                normalize(test, COLUMN)
                
            predictions = linear_regression_predict(test, base_norm, model_lr_norm, "base_norm")                
            sample_prediction['target'] = predictions[PREDICTION_COLUMN]                
            
            env.predict(sample_prediction)
            
        elif SUBMIT_MODEL == 'simple_prophet':
            print(f"Submitting: {SUBMIT_MODEL}...")
            COLUMNS = ['index','row_id','stock_id','ds']
            predictions = predict_simple_prophet(test, COLUMNS, "simple_prophet")
                    
    
            sample_prediction['target'] = predictions[PREDICTION_COLUMN]                    
            env.predict(sample_prediction)

    
    print("\n\nFinished submitting models")

In [27]:
for MODEL in MODELS:
    print(MODEL)

simple_prophet
lin_regr_base
lin_regr_base_norm


In [ ]:
# Change this so that you submit just on model so submit MODEL['simple_prophet']
submit("simple_prophet")

Model: simple_prophet selected for submission...
This version of the API is not optimized and should not be used to estimate the runtime of your code on the hidden test set.
Submitting: simple_prophet...
Configuring dataset for Prophet...
Progress... 

Predict: STOCK_ID: 0 with model: <prophet.forecaster.Prophet object at 0x7c383665f250>
Predict: STOCK_ID: 25 with model: <prophet.forecaster.Prophet object at 0x7c374051ba30>
Predict: STOCK_ID: 50 with model: <prophet.forecaster.Prophet object at 0x7c37f35a4370>
Predict: STOCK_ID: 75 with model: <prophet.forecaster.Prophet object at 0x7c379576c280>
Predict: STOCK_ID: 100 with model: <prophet.forecaster.Prophet object at 0x7c3790480be0>
Predict: STOCK_ID: 125 with model: <prophet.forecaster.Prophet object at 0x7c3742024160>
Predict: STOCK_ID: 150 with model: <prophet.forecaster.Prophet object at 0x7c378779bee0>
Predict: STOCK_ID: 175 with model: <prophet.forecaster.Prophet object at 0x7c37846c94b0>
Predict: STOCK_ID: 199 with model: <prop